# Project · Handwritten digits: PCA, softmax and optimisers from scratch

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app). 1,797 handwritten digits, 8×8 pixels (bundled with scikit-learn). Compress them with PCA via SVD, build a softmax classifier with hand-derived gradients, check those gradients numerically, race GD, momentum and Adam, and find the digits the model is least sure about using entropy. Uses Murphy ch. 2, 6, 7, 8 and 10.

Work through the milestones in order; each builds on the previous cell's variables. Solutions are hidden: double-click a solution cell's title to reveal it.

## Milestone 1 · Load and look

Load the digits, scale pixels to 0–1, split 1,400 train / rest test, and print the class counts.

In [ ]:
# Your turn
import numpy as np
from sklearn.datasets import load_digits
d = load_digits()
X, y = d.data / 16.0, d.target
rng = np.random.default_rng(0); idx = rng.permutation(len(y))
Xtr, ytr, Xte, yte = X[idx[:1400]], y[idx[:1400]], X[idx[1400:]], y[idx[1400:]]
print(Xtr.shape, Xte.shape, np.bincount(ytr))

In [ ]:
# @title Solution 1 (double-click to show the code)
import numpy as np
from sklearn.datasets import load_digits
d = load_digits()
X, y = d.data / 16.0, d.target
rng = np.random.default_rng(0); idx = rng.permutation(len(y))
Xtr, ytr, Xte, yte = X[idx[:1400]], y[idx[:1400]], X[idx[1400:]], y[idx[1400:]]
print(Xtr.shape, Xte.shape, np.bincount(ytr))

<details><summary><b>Answer and explanation</b></summary>

Each image is 64 numbers (8×8 pixels, values 0–16). Dividing by 16 puts them in 0–1. Classes are roughly balanced, so accuracy is a fair first metric.

Expected output:
```
(1400, 64) (397, 64) [135 143 126 150 133 145 140 149 142 137]
```

</details>

## Milestone 2 · PCA by SVD

Centre the training images and compute the SVD. How many components keep 90% of the variance? Reconstruct a test image from 5, 15 and 30 components and print the errors.

In [ ]:
# Your turn
mean = Xtr.mean(0)
U, s, Vt = np.linalg.svd(Xtr - mean, full_matrices=False)
var = s ** 2 / np.sum(s ** 2)
k90 = ...  # TODO: components needed for 90% variance
print("components for 90% variance:", k90)
x = Xte[0] - mean
for k in [5, 15, 30]:
    rec = ...  # TODO: project onto top-k directions and back
    print(f"k={k:2d}  reconstruction error {np.linalg.norm(x - rec) / np.linalg.norm(x):.3f}")

In [ ]:
# @title Solution 2 (double-click to show the code)
mean = Xtr.mean(0)
U, s, Vt = np.linalg.svd(Xtr - mean, full_matrices=False)
var = s ** 2 / np.sum(s ** 2)
k90 = int(np.searchsorted(np.cumsum(var), 0.90) + 1)  # components needed for 90% variance
print("components for 90% variance:", k90)
x = Xte[0] - mean
for k in [5, 15, 30]:
    rec = Vt[:k].T @ (Vt[:k] @ x)  # project onto top-k directions and back
    print(f"k={k:2d}  reconstruction error {np.linalg.norm(x - rec) / np.linalg.norm(x):.3f}")

<details><summary><b>Answer and explanation</b></summary>

About 20 of the 64 directions hold 90% of the variance, so digit images live near a much smaller subspace. Reconstruction error drops quickly with the first components. This is the linear autoencoder from station 7 (Murphy §20.1, §7.5).

Expected output:
```
components for 90% variance: 21
k= 5  reconstruction error 0.473
k=15  reconstruction error 0.355
k=30  reconstruction error 0.159
```

</details>

## Milestone 3 · Softmax regression: loss and gradient

Write the softmax cross-entropy loss and its gradient for weights W (64×10) and bias b (10). Derive the gradient first: why is it Xᵀ(P − Y)/N?

In [ ]:
# Your turn
def softmax(Z):
    Z = Z - Z.max(1, keepdims=True)                     # log-sum-exp trick (ch. 2)
    E = np.exp(Z); return E / E.sum(1, keepdims=True)
def loss_grad(W, b, X, y):
    N = len(y); P = softmax(X @ W + b)
    loss = -np.mean(np.log(P[np.arange(N), y] + 1e-12))
    Y = np.eye(10)[y]
    dZ = ...  # TODO: prediction minus truth, averaged
    return loss, X.T @ dZ, dZ.sum(0)
W0, b0 = np.zeros((64, 10)), np.zeros(10)
L0, gW, gb = loss_grad(W0, b0, Xtr, ytr)
print(f"loss at zero weights {L0:.4f}  (log 10 = {np.log(10):.4f})")

In [ ]:
# @title Solution 3 (double-click to show the code)
def softmax(Z):
    Z = Z - Z.max(1, keepdims=True)                     # log-sum-exp trick (ch. 2)
    E = np.exp(Z); return E / E.sum(1, keepdims=True)
def loss_grad(W, b, X, y):
    N = len(y); P = softmax(X @ W + b)
    loss = -np.mean(np.log(P[np.arange(N), y] + 1e-12))
    Y = np.eye(10)[y]
    dZ = (P - Y) / N  # prediction minus truth, averaged
    return loss, X.T @ dZ, dZ.sum(0)
W0, b0 = np.zeros((64, 10)), np.zeros(10)
L0, gW, gb = loss_grad(W0, b0, Xtr, ytr)
print(f"loss at zero weights {L0:.4f}  (log 10 = {np.log(10):.4f})")

<details><summary><b>Answer and explanation</b></summary>

With p = softmax(z) and loss −log p_true, the derivative with respect to each score is p_k − y_k (y is one-hot): the same 'prediction minus truth' as logistic regression, one entry per class. Chain rule through z = XW + b gives ∂L/∂W = Xᵀ(P − Y)/N and ∂L/∂b = mean of (P − Y) (Murphy §10.3).

Expected output:
```
loss at zero weights 2.3026  (log 10 = 2.3026)
```

</details>

## Milestone 4 · Gradient check

Compare your gradient for 5 random weights against finite differences. They should agree to about 6 digits.

In [ ]:
# Your turn
rng = np.random.default_rng(1)
W = rng.normal(0, 0.1, (64, 10)); b = np.zeros(10)
_, gW, _ = loss_grad(W, b, Xtr, ytr)
for _ in range(5):
    i, j = rng.integers(64), rng.integers(10); h = 1e-5
    Wp = W.copy(); Wp[i, j] += h; Wm = W.copy(); Wm[i, j] -= h
    num = ...  # TODO: central difference
    print(f"W[{i:2d},{j}]  analytic {gW[i, j]: .7f}  numeric {num: .7f}")

In [ ]:
# @title Solution 4 (double-click to show the code)
rng = np.random.default_rng(1)
W = rng.normal(0, 0.1, (64, 10)); b = np.zeros(10)
_, gW, _ = loss_grad(W, b, Xtr, ytr)
for _ in range(5):
    i, j = rng.integers(64), rng.integers(10); h = 1e-5
    Wp = W.copy(); Wp[i, j] += h; Wm = W.copy(); Wm[i, j] -= h
    num = (loss_grad(Wp, b, Xtr, ytr)[0] - loss_grad(Wm, b, Xtr, ytr)[0]) / (2 * h)  # central difference
    print(f"W[{i:2d},{j}]  analytic {gW[i, j]: .7f}  numeric {num: .7f}")

<details><summary><b>Answer and explanation</b></summary>

If they disagree, the derivation or the code has a bug. This check is how you verify any hand-written backward pass before trusting it (station 4).

Expected output:
```
W[ 1,9]  analytic  0.0008288  numeric  0.0008288
W[ 7,3]  analytic  0.0012276  numeric  0.0012276
W[16,2]  analytic -0.0000277  numeric -0.0000277
W[ 6,1]  analytic  0.0023249  numeric  0.0023249
W[17,4]  analytic  0.0127648  numeric  0.0127648
```

</details>

## Milestone 5 · Race the optimisers

Train for 300 full-batch steps with GD (lr 0.5), momentum (lr 0.5, β 0.9) and Adam (lr 0.05). Print the training loss at steps 10, 100, 300 and the test accuracy.

In [ ]:
# Your turn
def train(opt, steps=300):
    W, b = np.zeros((64, 10)), np.zeros(10)
    vW, vb, mW, mb, sW, sb = [np.zeros_like(a) for a in (W, b, W, b, W, b)]
    hist = []
    for t in range(1, steps + 1):
        L, gW, gb = loss_grad(W, b, Xtr, ytr); hist.append(L)
        if opt == "gd": W -= 0.5 * gW; b -= 0.5 * gb
        elif opt == "momentum":
            vW = 0.9 * vW + gW; vb = ...  # TODO: keep 90% of the previous velocity
            W -= 0.5 * vW; b -= 0.5 * vb
        else:
            for p, g, m, s in ((W, gW, mW, sW), (b, gb, mb, sb)):
                m[:] = 0.9 * m + 0.1 * g; s[:] = 0.999 * s + 0.001 * g * g
                p -= 0.05 * (m / (1 - 0.9 ** t)) / (np.sqrt(s / (1 - 0.999 ** t)) + 1e-8)
    acc = np.mean(np.argmax(Xte @ W + b, 1) == yte)
    return W, b, hist, acc
models = {}
for opt in ["gd", "momentum", "adam"]:
    W, b, h, acc = train(opt); models[opt] = (W, b)
    print(f"{opt:8s} loss@10 {h[9]:.3f}  @100 {h[99]:.3f}  @300 {h[-1]:.3f}  test acc {acc:.3f}")

In [ ]:
# @title Solution 5 (double-click to show the code)
def train(opt, steps=300):
    W, b = np.zeros((64, 10)), np.zeros(10)
    vW, vb, mW, mb, sW, sb = [np.zeros_like(a) for a in (W, b, W, b, W, b)]
    hist = []
    for t in range(1, steps + 1):
        L, gW, gb = loss_grad(W, b, Xtr, ytr); hist.append(L)
        if opt == "gd": W -= 0.5 * gW; b -= 0.5 * gb
        elif opt == "momentum":
            vW = 0.9 * vW + gW; vb = 0.9 * vb + gb  # keep 90% of the previous velocity
            W -= 0.5 * vW; b -= 0.5 * vb
        else:
            for p, g, m, s in ((W, gW, mW, sW), (b, gb, mb, sb)):
                m[:] = 0.9 * m + 0.1 * g; s[:] = 0.999 * s + 0.001 * g * g
                p -= 0.05 * (m / (1 - 0.9 ** t)) / (np.sqrt(s / (1 - 0.999 ** t)) + 1e-8)
    acc = np.mean(np.argmax(Xte @ W + b, 1) == yte)
    return W, b, hist, acc
models = {}
for opt in ["gd", "momentum", "adam"]:
    W, b, h, acc = train(opt); models[opt] = (W, b)
    print(f"{opt:8s} loss@10 {h[9]:.3f}  @100 {h[99]:.3f}  @300 {h[-1]:.3f}  test acc {acc:.3f}")

<details><summary><b>Answer and explanation</b></summary>

Momentum and Adam reach a low loss much sooner than plain GD. All three end with test accuracy around 95–97%, which is good for a linear model on raw pixels.

Expected output:
```
gd       loss@10 1.598  @100 0.415  @300 0.227  test acc 0.960
momentum loss@10 0.631  @100 0.114  @300 0.069  test acc 0.977
adam     loss@10 0.562  @100 0.089  @300 0.034  test acc 0.977
```

</details>

## Milestone 6 · Which digits is it unsure about?

Using the Adam model, compute the entropy (in bits) of each test prediction. Print the 5 most uncertain test digits with their true label, predicted label and top-2 probabilities. What's the maximum possible entropy?

In [ ]:
# Your turn
W, b = models["adam"]
P = softmax(Xte @ W + b)
H = ...  # TODO: entropy of each prediction
for i in np.argsort(-H)[:5]:
    top = np.argsort(-P[i])[:2]
    print(f"true {yte[i]}  pred {top[0]}  H {H[i]:.2f} bits  p={P[i, top[0]]:.2f}/{P[i, top[1]]:.2f} (for {top[0]}/{top[1]})")
print("max possible entropy:", round(np.log2(10), 3))

In [ ]:
# @title Solution 6 (double-click to show the code)
W, b = models["adam"]
P = softmax(Xte @ W + b)
H = -np.sum(P * np.log2(P + 1e-12), axis=1)  # entropy of each prediction
for i in np.argsort(-H)[:5]:
    top = np.argsort(-P[i])[:2]
    print(f"true {yte[i]}  pred {top[0]}  H {H[i]:.2f} bits  p={P[i, top[0]]:.2f}/{P[i, top[1]]:.2f} (for {top[0]}/{top[1]})")
print("max possible entropy:", round(np.log2(10), 3))

<details><summary><b>Answer and explanation</b></summary>

Maximum entropy for 10 classes is log₂ 10 ≈ 3.32 bits (completely unsure). The most uncertain digits are genuinely ambiguous shapes. Here: a 9 that looks like a 5, a 2 torn between 2 and 8, a 3 that looks like a 5. Several of them are the model's mistakes. Entropy turns a probability vector into one 'how unsure' number (Murphy §6.1), and sorting by it is a cheap way to find the cases a human should check.

Expected output:
```
true 9  pred 5  H 2.16 bits  p=0.42/0.24 (for 5/1)
true 2  pred 2  H 1.74 bits  p=0.48/0.33 (for 2/8)
true 6  pred 6  H 1.61 bits  p=0.62/0.14 (for 6/0)
true 8  pred 1  H 1.48 bits  p=0.64/0.18 (for 1/6)
true 3  pred 5  H 1.46 bits  p=0.53/0.38 (for 5/3)
max possible entropy: 3.322
```

</details>

## Milestone 7 · Write-up

What did PCA tell you about the data, what does the gradient check guarantee, and why did Adam win the race?

✍️ *Write your answer here.*

<details><summary><b>Model answer</b></summary>

PCA showed the 64 pixels really vary along about 20 directions, so the data is far simpler than its raw size. The gradient check guarantees the backward pass matches the loss you wrote, not that the model is good. Adam won early because it scales each weight's step by that weight's typical gradient, so rarely active pixels and busy pixels both get sensible steps from the first iteration.

</details>